> 🔑 **BẢN LỜI GIẢI** — chỉ mở khi đã tự làm và vẫn bí.

# Diffusion · GĐ1 · Thực hành — Tự viết DDPM từ con số 0

Notebook này đi kèm trang lý thuyết **Diffusion tạo ảnh thế nào**. Ta làm diffusion trên **điểm 2D** thay vì ảnh: cùng thuật toán với Stable Diffusion, nhưng train chỉ mất khoảng 30 giây trên CPU và nhìn được bằng mắt.

| Phần | Mục lý thuyết | Bạn sẽ tự viết |
|---|---|---|
| A | 03 · Lịch nhiễu | `betas`, `alphas`, `alpha_bars` |
| B | 03 · Forward process | hàm `q_sample` thêm nhiễu nhảy thẳng tới bước t |
| C | 04 · Model dự đoán nhiễu | time embedding và mạng khử nhiễu |
| D | 05 · Train | vòng lặp train (Algorithm 1 của bài báo DDPM) |
| E | 06 · Sampling | vòng lặp sinh dữ liệu (Algorithm 2) |
| F | 07 · Thí nghiệm | bỏ nhiễu khi sampling, và bộ khử nhiễu "hoàn hảo" |

**Cách làm:** chạy từ trên xuống, viết code vào `# TODO`, chạy ô **✅ Kiểm tra** ngay sau đó.

**Ký hiệu:** trong code, bước thời gian `t` chạy từ `0` đến `T-1` (Python đếm từ 0). Trong bài báo và trang lý thuyết, t chạy từ 1 đến T. Hai cách chỉ lệch nhau 1, công thức giữ nguyên.

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)

def make_spiral(n=10000, seed=0):
    r = np.random.default_rng(seed)
    th = np.sqrt(r.uniform(0, 1, n)) * 3 * np.pi
    x = np.stack([th * np.cos(th), th * np.sin(th)], 1) / (3 * np.pi)
    x += r.normal(0, 0.02, x.shape)
    return torch.tensor(x, dtype=torch.float32)

data = make_spiral()
plt.figure(figsize=(4, 4)); plt.scatter(data[:3000, 0], data[:3000, 1], s=1); plt.axis("equal")
plt.title("Dữ liệu thật: 10.000 điểm hình xoắn ốc"); plt.show()
print("data:", tuple(data.shape))

Hãy coi mỗi điểm 2D như một "bức ảnh" chỉ có 2 pixel. Mục tiêu: model học được phân phối hình xoắn ốc, rồi **tự sinh ra điểm mới** nằm trên xoắn ốc, xuất phát từ nhiễu thuần túy. Với ảnh 512×512×3 thì mỗi "điểm" có 786.432 chiều, còn thuật toán giữ nguyên.

## Phần A — Lịch nhiễu (noise schedule)

DDPM dùng $T = 1000$ bước. Ở bước $t$ ta thêm một lượng nhiễu nhỏ $\beta_t$, tăng tuyến tính từ $10^{-4}$ đến $0.02$:

$$ \alpha_t = 1 - \beta_t \qquad \bar\alpha_t = \alpha_1 \cdot \alpha_2 \cdots \alpha_t $$

$\bar\alpha_t$ (đọc: alpha bar) cho biết **còn lại bao nhiêu tín hiệu gốc** sau t bước. Gợi ý: `torch.linspace`, `torch.cumprod`.

In [ ]:
T = 1000
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

In [ ]:
# ✅ Kiểm tra A
assert betas is not None and betas.shape == (T,), "betas phải có shape (1000,)"
assert abs(betas[0].item() - 1e-4) < 1e-9 and abs(betas[-1].item() - 0.02) < 1e-9, "betas phải đi từ 1e-4 đến 0.02"
assert torch.allclose(alphas, 1 - betas)
assert abs(alpha_bars[0].item() - 0.9999) < 1e-6, "alpha_bars[0] phải bằng alphas[0]"
assert abs(alpha_bars[-1].item() - 4.0358e-5) < 2e-6, "alpha_bars cuối phải ≈ 4e-5 (gần như mất hết tín hiệu)"
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(alpha_bars.sqrt(), label="√ᾱ  (hệ số tín hiệu)"); ax.plot((1 - alpha_bars).sqrt(), label="√(1−ᾱ)  (hệ số nhiễu)")
ax.set_xlabel("t"); ax.legend(); plt.show()
print(f"✅ A đúng! Ở t=500, tín hiệu còn √ᾱ = {alpha_bars[499].sqrt():.3f}; ở t=1000 còn {alpha_bars[-1].sqrt():.4f}")

## Phần B — Forward process: thêm nhiễu

Điều kỳ diệu của DDPM: không cần thêm nhiễu 500 lần để tới bước 500. Có thể **nhảy thẳng**:

$$ x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\, \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, I) $$

`t` là một tensor số nguyên shape (N,), mỗi điểm trong batch có thể ở một bước khác nhau. Bạn cần lấy `alpha_bars[t]` (shape (N,)) rồi đổi thành (N, 1) để nhân được với `x0` shape (N, 2).

In [ ]:
def q_sample(x0, t, eps):
    ab = alpha_bars[t].unsqueeze(1)          # (N,) → (N, 1)
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps

In [ ]:
# ✅ Kiểm tra B
x0 = torch.tensor([[2.0, -1.0]]); e = torch.tensor([[-1.0, 0.5]])
out = q_sample(x0, torch.tensor([499]), e)
ab = alpha_bars[499]
assert out.shape == (1, 2), f"shape phải là (1, 2), đang là {tuple(out.shape)}"
assert torch.allclose(out, ab.sqrt() * x0 + (1 - ab).sqrt() * e), "Công thức chưa đúng"
big = data[:5000]
xT = q_sample(big, torch.full((5000,), T - 1), torch.randn_like(big))
x1 = q_sample(big, torch.zeros(5000, dtype=torch.long), torch.randn_like(big))
assert abs(xT.mean()) < 0.05 and abs(xT.std() - 1) < 0.05, "Ở t cuối, x_T phải gần như nhiễu chuẩn N(0, 1)"
assert (x1 - big).abs().mean() < 0.02, "Ở t=0, x_t phải gần như dữ liệu gốc"
mixed = q_sample(big[:3], torch.tensor([0, 500, 999]), torch.zeros(3, 2))
assert torch.allclose(mixed, alpha_bars[[0, 500, 999]].sqrt().unsqueeze(1) * big[:3]), "Mỗi điểm phải dùng đúng t của nó"

fig, axes = plt.subplots(1, 6, figsize=(18, 3))
for ax, tt in zip(axes, [0, 50, 100, 250, 500, 999]):
    xt = q_sample(data[:3000], torch.full((3000,), tt), torch.randn(3000, 2))
    ax.scatter(xt[:, 0], xt[:, 1], s=1); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_title(f"t = {tt + 1}")
plt.show()
print("✅ B đúng! Xoắn ốc tan dần thành đám nhiễu tròn.")

## Phần C — Model dự đoán nhiễu

Model nhận **điểm đã nhiễu** $x_t$ và **bước t**, trả về dự đoán nhiễu $\hat\varepsilon$ đã được thêm vào. Model phải biết t vì ở t lớn thì gần như toàn nhiễu, ở t nhỏ thì chỉ còn chút nhiễu.

### C1. Time embedding
Đưa số nguyên t thẳng vào mạng không hiệu quả. Ta biến t thành một vector gồm sin và cos ở nhiều tần số khác nhau, giống positional encoding của Transformer:

$$ \text{emb}(t) = [\sin(t\,\omega_0), \dots, \sin(t\,\omega_{h-1}),\ \cos(t\,\omega_0), \dots, \cos(t\,\omega_{h-1})], \qquad \omega_i = 10000^{-i/h},\ h = \text{dim}/2 $$

In [ ]:
def time_embedding(t, dim=32):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half) / half)   # ω_i = 10000^(-i/h)
    args = t.float().unsqueeze(1) * freqs.unsqueeze(0)                # (N, half)
    return torch.cat([torch.sin(args), torch.cos(args)], dim=1)

In [ ]:
# ✅ Kiểm tra C1
emb = time_embedding(torch.tensor([0, 1, 999]))
assert emb.shape == (3, 32), f"shape phải là (3, 32), đang là {tuple(emb.shape)}"
assert torch.allclose(emb[0, :16], torch.zeros(16)) and torch.allclose(emb[0, 16:], torch.ones(16)), "t=0: sin = 0, cos = 1"
assert abs(emb[1, 0].item() - math.sin(1.0)) < 1e-6, "Tần số đầu tiên phải là ω_0 = 1"
assert abs(emb[2, 15].item() - math.sin(999 * 10000 ** (-15 / 16))) < 1e-5, "Tần số cuối chưa đúng"
plt.figure(figsize=(8, 3)); plt.imshow(time_embedding(torch.arange(0, 1000, 5)).T, aspect="auto", cmap="RdBu")
plt.xlabel("t / 5"); plt.ylabel("chiều embedding"); plt.show()
print("✅ C1 đúng! Mỗi t có một 'dấu vân tay' riêng: chiều trên đổi nhanh, chiều dưới đổi chậm.")

### C2. Mạng khử nhiễu
Một MLP đơn giản như Bài 1: nối $x_t$ (2 chiều) với embedding của t (32 chiều) thành vector 34 chiều, rồi qua

`Linear(34,128) → ReLU → Linear(128,128) → ReLU → Linear(128,128) → ReLU → Linear(128,2)`

Đầu ra 2 chiều là $\hat\varepsilon$. Với ảnh, MLP này được thay bằng U-Net (GĐ2), còn ý tưởng giữ nguyên.

In [ ]:
class Denoiser(nn.Module):
    def __init__(self, hidden=128, tdim=32):
        super().__init__()
        self.tdim = tdim
        self.net = nn.Sequential(
            nn.Linear(2 + tdim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 2),
        )

    def forward(self, x, t):
        return self.net(torch.cat([x, time_embedding(t, self.tdim)], dim=1))

In [ ]:
# ✅ Kiểm tra C2
m = Denoiser()
out = m(torch.randn(7, 2), torch.randint(0, T, (7,)))
n_params = sum(p.numel() for p in m.parameters())
assert out.shape == (7, 2), f"Output phải có shape (7, 2), đang là {tuple(out.shape)}"
assert n_params == 37762, f"Số tham số phải là 37.762, đang là {n_params:,}"
print(f"✅ C2 đúng! Mạng có {n_params:,} tham số. (Stable Diffusion 1.5: U-Net ~860 triệu)")

## Phần D — Train (Algorithm 1 của DDPM)

Mỗi bước:
1. Lấy ngẫu nhiên một batch 512 điểm thật $x_0$
2. Với mỗi điểm, chọn ngẫu nhiên một bước $t \in \{0, \dots, T-1\}$
3. Lấy nhiễu $\varepsilon \sim \mathcal{N}(0, I)$, tính $x_t$ bằng `q_sample`
4. Model đoán nhiễu: $\hat\varepsilon = \text{model}(x_t, t)$
5. Loss là MSE giữa nhiễu thật và nhiễu đoán: $\mathcal{L} = \|\varepsilon - \hat\varepsilon\|^2$
6. 5 bước quen thuộc: `zero_grad → backward → step` (forward và loss đã có ở trên)

Chỉ vậy. Train diffusion chỉ là **hồi quy MSE**, giống Phần D Bài 1. Chạy khoảng 15–40 giây.

In [ ]:
torch.manual_seed(0)
model = Denoiser()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
losses = []

for step in range(6000):
    idx = torch.randint(0, len(data), (512,))
    x0 = data[idx]
    t = torch.randint(0, T, (512,))
    eps = torch.randn_like(x0)
    x_t = q_sample(x0, t, eps)
    eps_hat = model(x_t, t)
    loss = F.mse_loss(eps_hat, eps)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

In [ ]:
# ✅ Kiểm tra D
assert len(losses) == 6000, "Mỗi bước phải ghi một giá trị loss"
late = float(np.mean(losses[-500:]))
plt.figure(figsize=(8, 3)); plt.plot(np.convolve(losses, np.ones(100) / 100, mode="valid")); plt.yscale("log")
plt.title("Loss (trung bình trượt 100 bước)"); plt.xlabel("bước"); plt.show()
assert losses[0] > 0.6, "Loss ban đầu phải ≈ 1 (model chưa train đoán ra ~0, còn nhiễu thật có phương sai 1)"
assert late < 0.2, f"Loss cuối trung bình {late:.3f} — phải < 0.2. Kiểm tra lại q_sample và thứ tự các bước."
print(f"✅ D đúng! Loss từ {losses[0]:.2f} xuống {late:.3f}.")
print("   Loss không về 0 được. Ô dưới đo loss riêng ở từng bước t để xem chỗ nào khó.")

In [ ]:
# Loss ở từng bước t (đã viết sẵn) — đoán trước: t nhỏ hay t lớn khó hơn?
with torch.no_grad():
    for tt in [0, 10, 100, 250, 500, 999]:
        x0 = data[:5000]; t = torch.full((5000,), tt); e = torch.randn_like(x0)
        print(f"t = {tt + 1:4d} | loss = {F.mse_loss(model(q_sample(x0, t, e), t), e).item():.3f}")

Kết quả có thể ngược với trực giác. Ở **t lớn**, $x_t$ gần như chính là nhiễu $\varepsilon$, nên đoán $\hat\varepsilon \approx x_t$ là gần đúng: dễ. Ở **t nhỏ**, lượng nhiễu thêm vào còn nhỏ hơn cả độ dày của nét xoắn ốc, model không phân biệt được đâu là nhiễu, đâu là dữ liệu: khó.

## Phần E — Sampling: sinh dữ liệu mới (Algorithm 2)

Bắt đầu từ nhiễu thuần $x_T \sim \mathcal{N}(0, I)$, đi ngược từ $t = T-1$ về $0$. Mỗi bước:

$$ x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left( x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\hat\varepsilon(x_t, t) \right) + \sigma_t z, \qquad z \sim \mathcal{N}(0, I),\ \ \sigma_t = \sqrt{\beta_t} $$

Ở bước cuối cùng (`t == 0`) **không** cộng nhiễu $z$.

Ta thêm tham số `add_noise` để làm thí nghiệm ở Phần F: nếu `add_noise=False` thì không bao giờ cộng $\sigma_t z$.

In [ ]:
@torch.no_grad()
def sample(model, n=2000, add_noise=True, return_traj=False):
    x = torch.randn(n, 2)
    traj = {T: x.clone()}
    for t in range(T - 1, -1, -1):
        tt = torch.full((n,), t)
        eps_hat = model(x, tt)
        x = (x - betas[t] / (1 - alpha_bars[t]).sqrt() * eps_hat) / alphas[t].sqrt()
        if t > 0 and add_noise:
            x = x + betas[t].sqrt() * torch.randn_like(x)
        if t in (750, 500, 250, 100, 0):
            traj[t] = x.clone()
    return (x, traj) if return_traj else x

In [ ]:
# ✅ Kiểm tra E
torch.manual_seed(1)
samples, traj = sample(model, return_traj=True)
ref = data[:5000]
def nn_dist(a, b):  # khoảng cách trung bình từ mỗi điểm của a tới điểm gần nhất của b
    return torch.cdist(a, b).min(1).values.mean().item()
precision = nn_dist(samples, ref)               # điểm sinh ra có nằm trên xoắn ốc không?
coverage = nn_dist(make_spiral(2000, 5), samples)  # có phủ hết xoắn ốc không?
baseline = nn_dist(torch.randn(2000, 2), ref)

fig, axes = plt.subplots(1, 6, figsize=(18, 3))
for ax, k in zip(axes, [T, 750, 500, 250, 100, 0]):
    ax.scatter(traj[k][:, 0], traj[k][:, 1], s=1, c="C1"); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
    ax.set_title("nhiễu thuần" if k == T else f"t = {k}")
plt.show()
print(f"precision {precision:.3f} | coverage {coverage:.3f} | (nhiễu ngẫu nhiên: {baseline:.3f})")
assert samples.shape == (2000, 2)
assert torch.isfinite(samples).all(), "Có nan/inf — kiểm tra lại công thức (chia đúng √α_t chưa?)"
assert precision < 0.06, "Điểm sinh ra chưa nằm trên xoắn ốc. Kiểm tra công thức sampling và dấu trừ."
assert coverage < 0.05, "Điểm sinh ra chưa phủ hết xoắn ốc."
print("✅ E đúng! Bạn vừa sinh dữ liệu mới từ nhiễu — đúng thuật toán của DDPM.")

## Phần F — Hai thí nghiệm (đã viết sẵn, bạn chạy và giải thích)

### F1. Bỏ nhiễu $\sigma_t z$ khi sampling thì sao?

In [ ]:
torch.manual_seed(1)
s_nonoise = sample(model, add_noise=False)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 4))
a1.scatter(samples[:, 0], samples[:, 1], s=1, c="C1"); a1.set_title("Có σ_t·z")
a2.scatter(s_nonoise[:, 0], s_nonoise[:, 1], s=1, c="C3"); a2.set_title("Không có σ_t·z")
for a in (a1, a2): a.set_xlim(-1.3, 1.3); a.set_ylim(-1.3, 1.3)
plt.show()
print(f"Độ tản (std) khi có nhiễu: {samples.std(0).numpy().round(3)} | khi không có nhiễu: {s_nonoise.std(0).numpy().round(3)}")

### F2. Bộ khử nhiễu "hoàn hảo" học thuộc dữ liệu

Với một tập dữ liệu hữu hạn, ta có thể tính **chính xác** đáp án tối ưu cho bài toán dự đoán nhiễu, không cần mạng nơ-ron. Đó là trung bình có trọng số của các điểm train, trọng số lớn cho điểm nào khớp với $x_t$ nhất. Ô dưới chạy sampling với bộ khử nhiễu hoàn hảo trên chỉ 40 điểm train.

In [ ]:
small = make_spiral(40, seed=11)

@torch.no_grad()
def ideal_eps(x, t):
    ab = alpha_bars[t]
    d2 = ((x[:, None, :] - ab.sqrt() * small[None]) ** 2).sum(-1)   # khoảng cách tới từng điểm train (đã co lại)
    w = torch.softmax(-d2 / (2 * (1 - ab)), dim=1)                   # điểm train nào khớp nhất → trọng số lớn
    x0_hat = w @ small                                                # đoán x0
    return (x - ab.sqrt() * x0_hat) / (1 - ab).sqrt()                # đổi đoán x0 thành đoán ε

torch.manual_seed(2)
x = torch.randn(500, 2)
for t in range(T - 1, -1, -1):
    x = (x - betas[t] / (1 - alpha_bars[t]).sqrt() * ideal_eps(x, t)) / alphas[t].sqrt()
    if t > 0: x = x + betas[t].sqrt() * torch.randn_like(x)

d = torch.cdist(x, small).min(1).values
plt.figure(figsize=(4, 4)); plt.scatter(x[:, 0], x[:, 1], s=20, c="C1", alpha=0.3, label="500 điểm sinh ra")
plt.scatter(small[:, 0], small[:, 1], s=12, c="k", marker="x", label="40 điểm train"); plt.legend(); plt.axis("equal"); plt.show()
print(f"Khoảng cách trung bình từ điểm sinh ra tới điểm train gần nhất: {d.mean():.4f}")

## Câu hỏi nộp bài

Gửi Claude câu trả lời bằng lời của bạn:

1. Ở Phần D, vì sao loss bắt đầu ở khoảng 1? (Gợi ý: model mới khởi tạo đoán $\hat\varepsilon \approx 0$, mà $\varepsilon \sim \mathcal{N}(0, 1)$.)
2. Ở F1, khi bỏ $\sigma_t z$, các điểm sinh ra trông thế nào? Vì sao nhiễu lại cần thiết khi *sinh* dữ liệu?
3. Ở F2, bộ khử nhiễu "hoàn hảo" sinh ra điểm mới hay chỉ chép lại điểm train? Vậy vì sao một model **không hoàn hảo** (mạng nơ-ron) lại tạo ra được ảnh mới mà Stable Diffusion chưa từng thấy?
4. Thử đổi `T = 50` (giữ betas từ 1e-4 tới 0.02), train lại và sample. Kết quả tốt hơn hay tệ hơn? Vì sao? (Gợi ý: xem `alpha_bars[-1]`.)

Tiếp theo: **GĐ2 · Diffusion cho ảnh** — thay MLP bằng U-Net và sinh ảnh Fashion-MNIST.